# ARIA Dataset - Quick Start

This notebook demonstrates how to use the ARIA library for loading and working with the dataset.

**ARIA** = Authentic vs Reconstructed Isolated Audio


In [ ]:
# Set your dataset path
ARIA_PATH = "/Volumes/Lexar/datasets/aria"  # Change this to your path

## 1. Loading the Dataset


In [ ]:
from aria import ARIADataset, check_dataset_exists

# Quick check
if check_dataset_exists(ARIA_PATH):
    print("Dataset found!")
else:
    print("Dataset not found. Update ARIA_PATH above.")


In [ ]:
# Load dataset
dataset = ARIADataset(ARIA_PATH)

print(f"Loaded {len(dataset)} tracks")
print(f"Genres: {dataset.genres}")
print(f"Available codecs: {dataset.codecs}")


In [ ]:
# Dataset statistics
stats = dataset.stats()
stats

## 2. Exploring Tracks


In [ ]:
# Get a track by index
track = dataset[0]

print(f"Track: {track.display_name}")
print(f"Genre: {track.genre}")
print(f"Stems: {track.stem_names}")
print(f"Number of stems: {track.num_stems}")
print(f"Track ID: {track.id}")


In [ ]:
# Filter tracks
rock_tracks = dataset.filter(genre="rock")
print(f"Rock tracks: {len(rock_tracks)}")

# Tracks with exactly 4 stems
four_stem_tracks = dataset.filter(min_stems=4, max_stems=4)
print(f"4-stem tracks: {len(four_stem_tracks)}")


## 3. Loading Audio


In [ ]:
from aria import load_audio
from aria.track import get_stem_path, get_available_codecs

track = dataset[0]

# Load a real stem
stem_name = track.stem_names[0]
real_path = get_stem_path(track, stem_name, source="real")
real_audio, sr = load_audio(real_path)

print(f"Loaded: {stem_name}")
print(f"Shape: {real_audio.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {real_audio.shape[1] / sr:.1f} seconds")


In [ ]:
# Load the AI-generated version of the same stem
ai_path = get_stem_path(track, stem_name, source="ai_generated", codec="encodec_3")
ai_audio, sr = load_audio(ai_path)

print(f"AI stem shape: {ai_audio.shape}")

In [ ]:
# Check ARIA audio data properties
print(f"Real audio - Shape: {real_audio.shape}, Dtype: {real_audio.dtype}")
print(f"Real audio - Min: {real_audio.min():.6f}, Max: {real_audio.max():.6f}")
print(f"AI audio - Shape: {ai_audio.shape}, Dtype: {ai_audio.dtype}")
print(f"AI audio - Min: {ai_audio.min():.6f}, Max: {ai_audio.max():.6f}")

In [ ]:
from IPython.display import Audio, display
display(Audio(real_audio, rate=sr))
display(Audio(ai_audio, rate=sr))

In [ ]:
print("Real stem (10 seconds fragment):")
display(Audio(real_audio[0][sr*10:sr*20], rate=sr))
print("AI generated stem (10 seconds fragment):")
display(Audio(ai_audio[0][sr*10:sr*20], rate=sr))


## 4. Creating Mixes On-the-Fly


In [ ]:
from aria import MixSpec, compute_mix, generate_all_mix_specs

track = dataset[0]

# Create mix specifications
# ai_indices specifies which stems should be AI-generated (0-indexed)

all_real = MixSpec(ai_indices=(), codec="encodec_3")
print(f"All real: {all_real.filename} -> {all_real.u_folder}")

one_ai = MixSpec(ai_indices=(0,), codec="encodec_3")
print(f"One AI:   {one_ai.filename} -> {one_ai.u_folder}")

two_ai = MixSpec(ai_indices=(0, 2), codec="encodec_3")
print(f"Two AI:   {two_ai.filename} -> {two_ai.u_folder}")


In [ ]:
# Compute a mix on-the-fly
import time

spec = MixSpec(ai_indices=(0, 2), codec="encodec_3")

t0 = time.time()
mix_audio = compute_mix(track, spec)
elapsed = time.time() - t0

print(f"Computed mix in {elapsed:.2f}s")
print(f"Mix shape: {mix_audio.shape}")
print(f"Duration: {mix_audio.shape[1] / 48000:.1f} seconds")


In [ ]:
# Generate all possible mix specifications
all_specs = generate_all_mix_specs(track.num_stems, codec="encodec_3")

print(f"Total combinations for {track.num_stems} stems: {len(all_specs)}")
print(f"(This equals 2^{track.num_stems} = {2**track.num_stems})")


In [ ]:
all_specs

In [ ]:
spec = MixSpec(ai_indices=(), codec="encodec_3")
mix_audio = compute_mix(track, spec)
mix_audio

In [ ]:
display(Audio(mix_audio, rate=sr))

## 5. PyTorch Integration


In [ ]:
from aria.torch import ARIATorchDataset

# Create PyTorch dataset
torch_dataset = ARIATorchDataset(
    path=ARIA_PATH,
    codec="encodec_3",
    segment_duration=5.0,      # 5-second segments
    label_mode="binary",       # 0 = all real, 1 = has AI
    mix_sampling="binary",     # Only U0 (all real) and Un (all AI)
)

print(f"Dataset size: {len(torch_dataset)} samples")


In [ ]:
# Get a sample
sample = torch_dataset[0]

print(f"Audio shape: {sample['audio'].shape}")
print(f"Label: {sample['label']} ({'has AI' if sample['label'] else 'all real'})")
print(f"Track ID: {sample['track_id']}")


In [ ]:
# Use with DataLoader
from torch.utils.data import DataLoader
from aria.torch import aria_collate_fn

loader = DataLoader(
    torch_dataset, 
    batch_size=4, 
    shuffle=True, 
    num_workers=0,
    collate_fn=aria_collate_fn
)
# Get one batch
batch = next(iter(loader))

print(f"Batch audio shape: {batch['audio'].shape}")  # (B, 2, T)
print(f"Batch labels: {batch['label']}")


## 6. Dataset Validation


In [ ]:
from aria import validate_dataset, get_dataset_summary

# Quick summary
summary = get_dataset_summary(ARIA_PATH)
print("Dataset Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")


In [ ]:
# Full validation (check first 5 tracks for speed)
result = validate_dataset(
    ARIA_PATH,
    check_real_stems=True,
    check_ai_stems=True,
    codecs=["encodec_3"],
    limit=5,
)

print(f"Validation: {result}")


## Summary

This notebook demonstrated:

1. **Loading** - `ARIADataset(path)` to load the dataset
2. **Exploring** - Access tracks by index, filter by genre/stems
3. **Audio** - Load real and AI-generated stems with `load_audio()`
4. **Mixing** - Create any combination with `MixSpec` and `compute_mix()`
5. **PyTorch** - `ARIATorchDataset` for training deepfake detectors
6. **Validation** - `validate_dataset()` to check integrity

For more details, see the [README](../README.md) and [scripts/README.md](../scripts/README.md).
